In [0]:
%skip ./_resources/00-project-setup

In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="NETSUITE",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="cost",
    choices=["discount", "sales", "cost"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

In [0]:
from pyspark.sql.functions import *

cdc_raw_data = spark.read.option('multiline', False).format('json').load(f'{staging}/FoodQuest/Netsuite/Cost/Dennys/Dennys SZR/2025/Jan/cost*.json').limit(1)
display(cdc_raw_data)

In [0]:
%sql create table if not exists fq_dev_catalog.bronze.cost TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.enableChangeDataFeed' = 'true', delta.autoOptimize.optimizeWrite = true, delta.autoOptimize.autoCompact = true);

In [0]:
import time
from pyspark.sql.functions import col, expr, current_timestamp, regexp_replace, lit, substring, substring_index, to_timestamp, size

bronzeDF = (spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", "json")
                .option('multiline', False)
                .option("cloudFiles.allowOverwrites", "true") #re-ingest files if they are overwritten or update
                .option("cloudFiles.inferColumnTypes", "true")
                .option("cloudFiles.schemaLocation", f'{checkpoint}/{source}/{domain}/streaming/schema_cost')
                .load(f'{staging}/FoodQuest/Netsuite/Cost/*/cost*.json')
            )
# display(bronzeDF)

(bronzeDF.withColumn("ingestion_ts", current_timestamp())
        .withColumn('file_name', substring_index(col('_metadata.file_name'), '.', 1))
        .withColumn(
                "effective_from_date",
                to_date(split(col("file_name"), "_")[1], "yyyyMMdd")
        ).withColumn(
                "effective_to_date",
                to_date(split(col("file_name"), "_")[2], "yyyyMMdd")
        )
        .withColumn('file_path', regexp_replace(col('_metadata.file_path'), '%20', ' '))
        # .withColumn(
        #         "deployment_name",
        #         split(col("file_path"), "/")[6]
        # )
        .withColumn('sys_id', expr('uuid()'))
        .writeStream
        .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_cost2')
        .trigger(availableNow=True)
        .queryName(f'{domain}WriteStream')
        .outputMode('append')
        .toTable(f"`{environment}_catalog`.`bronze`.`{domain}`", mergeSchema=True)
).awaitTermination()

time.sleep(20)

In [0]:
%sql
SELECT 
  count(distinct effective_from_date) as week_cardinality,
  COUNT(*) AS total_rows
FROM fq_dev_catalog.bronze.cost;

In [0]:
%sql select * from fq_dev_catalog.bronze.cost limit 1

In [0]:
for query in spark.streams.active:
    query.stop()